# LLM Fine-Tuning Pipeline - Interactive Demo

This notebook demonstrates the complete fine-tuning pipeline with step-by-step explanations.

In [ ]:
# Setup
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))

import torch
from src import (
    DataPreparer,
    ModelManager,
    FineTuningTrainer,
    ModelEvaluator,
    BaselineComparer,
)
from config import ModelConfig, TrainingConfig, EvaluationConfig

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Data Preparation

Let's start by preparing our dataset. We'll use a sample dataset for demonstration.

In [ ]:
# Initialize data preparer
preparer = DataPreparer(
    tokenizer_name="gpt2",
    max_length=128,
    text_column="text",
)

# Create sample dataset
print("Creating sample dataset...")
dataset = preparer.create_sample_dataset(num_samples=500)

print(f"\nDataset splits:")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

# Show sample
print(f"\nSample data:")
print(dataset['train'][0])

In [ ]:
# Tokenize dataset
print("Tokenizing dataset...")
tokenized_dataset = preparer.prepare_dataset(dataset)

print(f"\nTokenized dataset features: {tokenized_dataset['train'].column_names}")
print(f"Sample token IDs: {tokenized_dataset['train'][0]['input_ids'][:10]}...")
print(f"Decoded: {preparer.decode(tokenized_dataset['train'][0]['input_ids'][:20])}")

## 2. Model Setup with LoRA

Now let's load the model and apply LoRA for parameter-efficient fine-tuning.

In [ ]:
# Configure model with LoRA
model_config = ModelConfig(
    model_name="gpt2",
    peft_method="lora",
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
)

# Create model manager
manager = ModelManager(model_config)

# Load model and tokenizer
model, tokenizer = manager.prepare_for_training()

# Print model info
info = manager.get_model_info()
print("\nModel Information:")
print(f"  Model: {info['model_name']}")
print(f"  Total parameters: {info['total_parameters']:,}")
print(f"  Trainable parameters: {info['trainable_parameters']:,}")
print(f"  Trainable %: {info['trainable_percentage']:.2f}%")

In [ ]:
# Test generation before training
test_prompt = "This product is"
inputs = tokenizer(test_prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nBefore training:")
print(f"  Prompt: {test_prompt}")
print(f"  Generated: {generated}")

## 3. Training

Now let's fine-tune the model on our dataset.

In [ ]:
# Configure training
training_config = TrainingConfig(
    output_dir="../outputs/notebook_demo",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_total_limit=1,
    fp16=False,
    gradient_checkpointing=True,
)

# Create trainer
trainer = FineTuningTrainer(
    model=model,
    tokenizer=tokenizer,
    config=training_config,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

print("Training configuration:")
print(f"  Epochs: {training_config.num_train_epochs}")
print(f"  Batch size: {training_config.per_device_train_batch_size}")
print(f"  Learning rate: {training_config.learning_rate}")

In [ ]:
# Train the model
print("Starting training...\n")
results = trainer.train()

print(f"\nTraining complete!")
print(f"Final training loss: {results['training_loss']:.4f}")

In [ ]:
# Test generation after training
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nAfter training:")
print(f"  Prompt: {test_prompt}")
print(f"  Generated: {generated}")

## 4. Evaluation

Let's evaluate our fine-tuned model and compare it with the baseline.

In [ ]:
# Evaluate fine-tuned model
eval_config = EvaluationConfig(
    batch_size=8,
    num_samples=20,
    max_new_tokens=30,
)

evaluator = ModelEvaluator(model, tokenizer, eval_config)

# Compute perplexity
perplexity = evaluator.compute_perplexity(tokenized_dataset["test"])
print(f"\nFine-tuned model perplexity: {perplexity:.2f}")

In [ ]:
# Compare with baseline
print("\nComparing with baseline model...\n")

comparer = BaselineComparer(
    fine_tuned_model=model,
    base_model_name="gpt2",
    tokenizer=tokenizer,
    config=eval_config,
)

eval_prompts = [
    "This product is",
    "I think this",
    "The quality is",
    "My experience was",
    "I would recommend",
]

comparison = comparer.compare(
    eval_dataset=tokenized_dataset["test"],
    prompts=eval_prompts,
)

# Generate report
report = comparer.generate_report(comparison)
print(report)

In [ ]:
# Show generation comparisons
print("\nGeneration Comparisons:")
print("=" * 60)

for sample in comparison.get("generation_comparison", [])[:5]:
    print(f"\nPrompt: {sample['prompt']}")
    print(f"  Fine-tuned: {sample['fine_tuned_output'][:80]}...")
    print(f"  Baseline:   {sample['baseline_output'][:80]}...")

## 5. Save Model

Save the fine-tuned model for later use.

In [ ]:
# Save the model
output_dir = "../outputs/final_model"
trainer.save_model(output_dir)
trainer.save_results(f"{output_dir}/training_results.json")

print(f"Model saved to: {output_dir}")
print(f"\nTo load the model later:")
print(f"  from transformers import AutoModelForCausalLM")
print(f"  from peft import PeftModel")
print(f"  base = AutoModelForCausalLM.from_pretrained('gpt2')")
print(f"  model = PeftModel.from_pretrained(base, '{output_dir}')")

## Summary

In this notebook, we demonstrated:

1. **Data Preparation**: Loading and tokenizing text data
2. **Model Setup**: Loading GPT-2 with LoRA for efficient fine-tuning
3. **Training**: Fine-tuning the model with gradient accumulation
4. **Evaluation**: Comparing fine-tuned vs baseline performance
5. **Saving**: Preserving the model for later use

### Key Takeaways

- LoRA enables fine-tuning with only ~0.2% of parameters trainable
- Fine-tuning typically improves perplexity by 20-40%
- The trained adapter is small and can be easily shared